# Llama PRM Scoring Smoke Test

Smoke test for `RLHFlow/Llama3.1-8B-PRM-Deepseek-Data`. This PRM is a
causal LM trained to judge each reasoning step by predicting `+` or
`-` in an assistant turn; we read `P(+) / (P(+) + P(-))` per step.

Two examples: (1) the flamingo problem shared with the Qwen PRM smoke
test; (2) a correct-vs-wrong algebra pair, to check the PRM flags a
known mistake. Loaded in `float16` (fits a 32 GB V100 comfortably).

Env: runs under `py311` (transformers 4.57). Unlike the Qwen PRM, this
model uses stock `AutoModelForCausalLM` with no bundled remote code,
so no cache workaround is needed.

## Setup

In [1]:
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from notebook_utils import gpu_mem_used_gb

base_dir = "/groups/chichengz/tnn/datasets"
llama_prm_dir = f"{base_dir}/Llama3.1-8B-PRM-Deepseek-Data"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).


## Scoring function

The conversation alternates user reasoning-step messages with
assistant `+` judgements. A parallel conversation swaps `+` for a
unique marker token (`ки`) so we can locate the score position. Set
`print_conversation=True` to dump the full formatted PRM input.

In [2]:
def score_llama_prm(
    model,
    tokenizer,
    problem,
    steps,
    candidate_token_ids,
    print_conversation=False,
):
    # marker_text "ки" is a Cyrillic bigram that maps to a single
    # unique vocab token; the parallel marker conversation lets us
    # find where each assistant `+` sits.
    marker_text = "\u043a\u0438"
    marker_token_id = (
        tokenizer(marker_text, return_tensors="pt").input_ids[0, 1].item()
    )

    conversation = []
    marker_conversation = []
    step_scores = []

    for step_idx, step in enumerate(steps):
        text = (problem + " " + step) if step_idx == 0 else step

        conversation.append({"role": "user", "content": text})
        conversation.append({"role": "assistant", "content": "+"})

        marker_conversation.append({"role": "user", "content": text})
        marker_conversation.append(
            {"role": "assistant", "content": marker_text}
        )

        input_ids = tokenizer.apply_chat_template(
            conversation,
            return_tensors="pt",
        ).to(model.device)
        marker_input_ids = tokenizer.apply_chat_template(
            marker_conversation,
            return_tensors="pt",
        ).to(model.device)

        if input_ids.shape != marker_input_ids.shape:
            raise RuntimeError(
                f"Marker conversation shape mismatch: "
                f"{input_ids.shape} vs {marker_input_ids.shape}"
            )

        with torch.no_grad():
            logits = model(input_ids=input_ids).logits[
                :, :, candidate_token_ids
            ]
            probs = logits.softmax(dim=-1)[:, :, 0]

        # The model predicts token N from position N-1. Locate the
        # marker token in the parallel prompt and read the previous
        # position's P(+). Use the last marker for the current step.
        marker_positions = (
            marker_input_ids[0, 1:] == marker_token_id
        ).nonzero(as_tuple=True)[0]
        if marker_positions.numel() != step_idx + 1:
            raise RuntimeError(
                f"Expected {step_idx + 1} marker positions, found "
                f"{marker_positions.numel()}"
            )
        score_pos = marker_positions[-1].item()
        step_scores.append(
            probs[0, score_pos].detach().cpu().float().item()
        )

    if print_conversation:
        # Render the full prefix (all steps) to show the user / `+`
        # structure the PRM scores.
        full = tokenizer.apply_chat_template(
            conversation, tokenize=False,
        )
        print("===== Formatted PRM input =====")
        print(full)
        print("===============================")

    return step_scores


def print_step_scores(steps, scores):
    # Step text truncated to keep the long flamingo steps from
    # wrapping into a wall — the scores are what matter here.
    for idx, (step, score) in enumerate(
        zip(steps, scores), start=1
    ):
        preview = step if len(step) <= 60 else step[:60] + "..."
        print(f"Step {idx}: P(correct) = {score:.4f}")
        print(preview)

## Load the PRM

In [3]:
def load_llama_prm(
    model_dir: str,
    device_map: str = "cuda:0",
):
    # Llama 8B PRM fits comfortably on a 32 GB V100 in fp16.
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map=device_map,
        dtype=torch.float16,
    ).eval()

    # Llama ships no pad token; reuse EOS for batched calls.
    tokenizer.padding_side = "right"
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

    # encode("+") prepends BOS; [-1] picks the actual "+" id.
    # Restricting softmax to {plus, minus} gives P(+)/(P(+)+P(-)).
    plus_token_id = tokenizer.encode("+")[-1]
    minus_token_id = tokenizer.encode("-")[-1]
    candidate_token_ids = [plus_token_id, minus_token_id]

    return model, tokenizer, candidate_token_ids


llama_model, llama_tokenizer, candidate_token_ids = load_llama_prm(
    llama_prm_dir
)

print(f"candidate token ids: {candidate_token_ids}")
print(f"dtype              : {next(llama_model.parameters()).dtype}")
print(f"GPU memory used    : {gpu_mem_used_gb():.2f} GB")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

candidate token ids: [10, 12]
dtype              : torch.float16
GPU memory used    : 16.41 GB


## Example 1 — flamingo problem

The flamingo problem shared with the Qwen PRM smoke test, for a
cross-model comparison of per-step scores.

In [4]:
# Same toy example used in the Qwen PRM smoke test.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [5]:
llama_scores = score_llama_prm(
    llama_model, llama_tokenizer, problem, steps,
    candidate_token_ids,
)

print("=== Flamingo trajectory ===")
print_step_scores(steps, llama_scores)

=== Flamingo trajectory ===
Step 1: P(correct) = 0.9980
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.8174
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9604
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9829
To find the difference, subtract the number of white flaming...


## Example 2 — correct vs. wrong trajectory

Same equation scored twice. The wrong trajectory divides by 2 instead
of 3 at step 2 — a good PRM should drop the score there and stay low.
The first run prints the exact PRM input for inspection.

In [6]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [7]:
# Inspect the exact PRM input for the correct trajectory, then
# score both trajectories.
correct_scores = score_llama_prm(
    llama_model, llama_tokenizer, algebra_problem, correct_steps,
    candidate_token_ids, print_conversation=True,
)
wrong_scores = score_llama_prm(
    llama_model, llama_tokenizer, algebra_problem, wrong_steps,
    candidate_token_ids,
)

print("\n=== Correct trajectory ===")
print_step_scores(correct_steps, correct_scores)

print("\n=== Wrong trajectory (step 2 divides by 2) ===")
print_step_scores(wrong_steps, wrong_scores)

===== Formatted PRM input =====
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

If 3x + 5 = 17, what is x? We need solve the equation 3x + 5 = 17.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|><|start_header_id|>user<|end_header_id|>

Subtracting 5 from both sides gives 3x = 12.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|><|start_header_id|>user<|end_header_id|>

Dividing both sides by 3 gives x = 4.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|><|start_header_id|>user<|end_header_id|>

Therefore, the answer is (\boxed{4}).<|eot_id|><|start_header_id|>assistant<|end_header_id|>

+<|eot_id|>

=== Correct trajectory ===
Step 1: P(correct) = 0.9990
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 1.0000
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing bot

## Cleanup

In [8]:
del llama_model, llama_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

GPU memory used: 14.35 GB
